In [72]:
import pandas as pd
import xlwings as xw
import openpyxl
import datetime
import os
import glob

In [73]:
zek103_dtypes = {
    'Werk': 'string',
    'Mat': 'string',
}

zek103_file_path = r"\\rfmesrv5\connect\DST_SAP_Transfer\P11\PPS_LUB\05_PURCHASING_AUTOMATION\ZEK103_PUR_LUB_002.xlsx"
main_excel_path = r'P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\PurchAutomationTemplate.xlsx'
supplier_files_directory_path = r'P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files'

In [61]:
def get_supplier_sap_numbers(filepath, sheet_name=None):
    """
    Returns a list of SAP numbers from an Excel file, which are found in column A
    between the row containing 'SAP' and the row containing 'Consumption'.

    :param filepath: Path to the Excel file (.xlsx)
    :param sheet_name: Name of the worksheet (if None, the active sheet is used)
    :return: List of SAP numbers (values from column A)
    """
    # Open the Excel file
    wb = openpyxl.load_workbook(filepath, data_only=True)
    if sheet_name:
        sheet = wb[sheet_name]
    else:
        sheet = wb.active

    # Search for the headers in column A
    row_sap = None
    row_consumption = None
    for row in range(1, sheet.max_row + 1):
        cell_value = sheet.cell(row=row, column=1).value
        if cell_value is not None:
            value_str = str(cell_value).strip().upper()
            if value_str == 'SAP' and row_sap is None:
                row_sap = row
            elif value_str == 'CONSUMPTION' and row_consumption is None:
                row_consumption = row
        if row_sap is not None and row_consumption is not None:
            break

    if row_sap is None or row_consumption is None:
        return []  # Or raise an Exception if you prefer

    # Get numbers between the found headers (exclusive)
    sap_numbers = []
    for row in range(row_sap + 1, row_consumption):
        value = sheet.cell(row=row, column=1).value
        if value is not None:
            sap_numbers.append(str(value))
    return sap_numbers

# Example usage:
sap_list = get_supplier_sap_numbers(main_excel_path)
print(sap_list)

['773633', '563088', '637682', '773550', '788701', '553330', '773551', '773552']


In [88]:
def get_zek103_data(zek103_data, plant, sap_numbers):
    zek103_data = zek103_data[zek103_data['Werk'].isin(plant)]
    zek103_data = zek103_data[zek103_data['Mat'].isin(sap_numbers)]

    zek103_data_grouped = zek103_data.groupby(['Lieferdatum', 'Mat'], as_index=False)['Best-Mg'].sum()

    zek103_data_grouped['Lieferdatum'] = pd.to_datetime(zek103_data_grouped['Lieferdatum'])  # opcjonalna konwersja na datę
    zek103_data_grouped['delayed'] = zek103_data_grouped['Lieferdatum'] < pd.Timestamp('today').normalize()

    return zek103_data_grouped

In [89]:
zek103_content = pd.read_excel(zek103_file_path, dtype=zek103_dtypes)
zek103_output = get_zek103_data(zek103_content, ['2101'], sap_list)
zek103_output

,Lieferdatum,Mat,Best-Mg,delayed
0,2026-01-28,563088,100.0,True
1,2026-02-04,563088,100.0,False
2,2026-02-17,563088,211.0,False
3,2026-02-17,773633,100.0,False


In [90]:
def update_excel_with_quantities(filepath, df, header_upper_bound, header_lower_bound, is_order_data=False):
    """
    Updates an Excel file with quantities from a DataFrame based on matching SAP numbers
    and Lieferdatum (dates).

    :param filepath: Path to the Excel file
    :param df: DataFrame with columns ['Lieferdatum', 'Mat', 'Best-Mg', 'delayed']
    """

    # Load the workbook
    wb = openpyxl.load_workbook(filepath)
    sheet = wb.active  # Operate on the active sheet

    # Find the row range for SAP numbers (between "SAP" and "Consumption")
    sap_start_row = None
    sap_end_row = None
    for row in range(1, sheet.max_row + 1):
        cell_value = sheet.cell(row=row, column=1).value  # Column A
        if cell_value is not None:
            value_str = str(cell_value).strip().upper()
            if value_str == header_upper_bound and sap_start_row is None:
                sap_start_row = row + 1  # Start after "SAP"
            elif value_str == header_lower_bound and sap_end_row is None:
                sap_end_row = row - 1  # End before "Consumption"

    if sap_start_row is None or sap_end_row is None:
        raise ValueError(f"Unable to find {header_upper_bound} and {header_lower_bound} headers in column A.")

    # Read SAP numbers from the specified range in column A
    sap_numbers = {}
    for row in range(sap_start_row, sap_end_row + 1):
        mat_number = sheet.cell(row=row, column=1).value
        if mat_number is not None:
            sap_numbers[str(mat_number)] = row

    # Read dates from the first row (headers starting from column M)
    date_columns = {'delayed': 12}  # Column with delayed orders

    for col in range(13, sheet.max_column + 1):  # Column index starts at M (13th column)
        date_value = sheet.cell(row=1, column=col).value
        if isinstance(date_value, pd.Timestamp) or isinstance(date_value, datetime.date):
            date_value = pd.Timestamp(date_value)  # Ensure it is a pandas Timestamp
        if date_value:
            date_columns[date_value.date()] = col

    # Clear data in rows between 'SAP' and 'Consumption', from column L to column L+300 (12th to 312th column)
    for row in range(sap_start_row, sap_end_row + 1):
        for col in range(12, 312):  # Column L (12th column) to 312th column
            sheet.cell(row=row, column=col).value = None

    # Iterate over the DataFrame rows
    for _, row in df.iterrows():
        lieferdatum = row['Lieferdatum']
        mat_number = row['Mat']
        quantity = row['Best-Mg']
        delayed = row['delayed']

        # Ensure lieferdatum is a date (convert if necessary)
        if isinstance(lieferdatum, str):
            lieferdatum = pd.to_datetime(lieferdatum).date()
        elif isinstance(lieferdatum, pd.Timestamp):
            lieferdatum = lieferdatum.date()

        # Match SAP number and date to find the correct cell
        if mat_number in sap_numbers and lieferdatum in date_columns:
            sap_row = sap_numbers[mat_number]
            date_col = date_columns[lieferdatum]
            if delayed and is_order_data:
                date_col = date_columns['delayed']

            # Write quantity to the matched cell
            sheet.cell(row=sap_row, column=date_col).value = quantity

    # Save the updated workbook
    wb.save(filepath)
    print(f"Excel file {os.path.basename(file_path)} updated successfully.")

In [91]:
def list_excel_files(directory):
    """
    Returns a list of full paths to all Excel files (.xlsx and .xlsm) in a specified directory.

    :param directory: Path to the directory to search for Excel files
    :return: List of full paths to Excel files
    """

    # Use glob to find .xlsx and .xlsm files in the directory
    xlsx_files = glob.glob(os.path.join(directory, '*.xlsx'))
    xlsm_files = glob.glob(os.path.join(directory, '*.xlsm'))

    # Combine the two lists
    excel_files = xlsx_files + xlsm_files

    return excel_files

In [92]:
# Example usage
supplier_files_paths = list_excel_files(supplier_files_directory_path)

# Print the full paths of all found Excel files
print("Excel files found:")
for file_path in supplier_files_paths:
    print(file_path)

Excel files found:
P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_dostawca1.xlsx
P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_dostawca2.xlsx


In [99]:
zek103_content = pd.read_excel(zek103_file_path, dtype=zek103_dtypes)
supplier_files_paths = list_excel_files(supplier_files_directory_path)

for file_path in supplier_files_paths:
    sap_list = get_supplier_sap_numbers(file_path)
    zek103_output = get_zek103_data(zek103_content, ['2101'], sap_list)
    update_excel_with_quantities(file_path, zek103_output, 'SAP', 'CONSUMPTION', True)

Excel file PurchAutomation_dostawca1 — kopia.xlsx updated successfully.
Excel file PurchAutomation_dostawca1.xlsx updated successfully.
Excel file PurchAutomation_dostawca2 — kopia.xlsx updated successfully.
Excel file PurchAutomation_dostawca2.xlsx updated successfully.
Excel file PurchAutomation_dostawca3 — kopia.xlsx updated successfully.
Excel file PurchAutomation_dostawca3.xlsx updated successfully.
Excel file PurchAutomation_dostawca4 — kopia.xlsx updated successfully.
Excel file PurchAutomation_dostawca4.xlsx updated successfully.
Excel file PurchAutomation_dostawca5 — kopia.xlsx updated successfully.
Excel file PurchAutomation_dostawca5.xlsx updated successfully.
